In [1]:
import os
import sys
import copy
import pickle
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

d:\git_repos\capstone\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\git_repos\capstone\.venv\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
BASE_DIR = r"d:\git_repos\capstone\graph-transformer"
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
GRAPH_DATA_PATH = os.path.join(OUTPUT_DIR, "graphs.pt")
VOCAB_PATH = os.path.join(OUTPUT_DIR, "vocab.pkl")
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "best_model.pt")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
BATCH_SIZE = 8
NUM_EPOCHS = 30
LEARNING_RATE = 5e-4
VAL_FRACTION = 0.15
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
from model import GraphTransformerAutoencoder, ModelConfig, reconstruction_loss

In [5]:
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [6]:
def split_data(graphs: list, val_fraction: float = VAL_FRACTION, seed: int = SEED):
  
    rng = random.Random(seed)

    ambient_graphs = [g for g in graphs if g.y.item() == 0 and str(g.capture_name).startswith("ambient_")]
    attack_graphs = [g for g in graphs if g.y.item() == 1 or not str(g.capture_name).startswith("ambient_")]

    by_capture = {}
    for g in ambient_graphs:
        by_capture.setdefault(g.capture_name, []).append(g)

    ambient_captures = sorted(list(by_capture.keys()))
    rng.shuffle(ambient_captures)

    n_val = max(1, int(len(ambient_captures) * val_fraction))
    val_captures = ambient_captures[:n_val]
    remaining = ambient_captures[n_val:]

    n_test = max(1, int(len(remaining) * 0.15))
    test_ambient_captures = remaining[:n_test]
    train_captures = remaining[n_test:]

    train_benign = [g for c in train_captures for g in by_capture[c]]
    val_benign = [g for c in val_captures for g in by_capture[c]]
    test_benign = [g for c in test_ambient_captures for g in by_capture[c]]
    test_set = test_benign + attack_graphs

    attack_captures = sorted(list({g.capture_name for g in attack_graphs}))

    print(f"Split captures: {len(train_captures)} train ambient, {len(val_captures)} val ambient, "
          f"{len(test_ambient_captures)} test ambient, {len(attack_captures)} attack captures.")
    print(f"Split graphs: {len(train_benign)} train (ambient), {len(val_benign)} val (ambient), "
          f"{len(test_set)} test total.")

    return train_benign, val_benign, test_set, train_captures, val_captures, test_ambient_captures

In [7]:
def align_graph_feature_dim(graphs, expected_dim: int):
    aligned = 0
    for g in graphs:
        if g.x.size(-1) != expected_dim:
            if g.x.size(-1) < expected_dim:
                pad = torch.zeros((g.x.size(0), expected_dim - g.x.size(-1)), dtype=g.x.dtype)
                g.x = torch.cat([g.x, pad], dim=1)
            else:
                g.x = g.x[:, :expected_dim]
            aligned += 1
    if aligned:
        print(f"Aligned {aligned} graphs to feature dimension {expected_dim}.")

In [8]:
def normalize_graph_features(train_graphs, val_graphs, test_graphs):  
    if not train_graphs:
        raise ValueError("Training split is empty; cannot compute feature normalization stats.")

    train_x = torch.cat([g.x for g in train_graphs], dim=0)
    mean = train_x.mean(dim=0)
    std = train_x.std(dim=0, unbiased=False)
    std = torch.where(std < 1e-6, torch.ones_like(std), std)

    for graph_list in (train_graphs, val_graphs, test_graphs):
        for g in graph_list:
            g.x = ((g.x - mean) / std).to(torch.float32)

    return mean, std

In [9]:
def apply_saved_normalization(graphs: list, mean: torch.Tensor, std: torch.Tensor):
    mean = mean.to(torch.float32)
    std = std.to(torch.float32)
    for g in graphs:
        g.x = ((g.x - mean) / std).to(torch.float32)

In [10]:
def run_epoch(model, loader, optimizer, config, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    n_graphs = 0

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for batch in loader:
            batch = batch.to(DEVICE)
            outputs = model(
                x_stats=batch.x,
                id_idx=batch.id_idx,
                edge_index=batch.edge_index,
                edge_weight=batch.edge_attr,
            )
            loss = reconstruction_loss(outputs, batch.x, batch.edge_index, config, model=model)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * batch.num_graphs
            n_graphs += batch.num_graphs

    return total_loss / max(n_graphs, 1)

In [11]:
def _rank_data_numpy(a: np.ndarray) -> np.ndarray:
    a = np.asarray(a)
    n = len(a)
    if n == 0:
        return np.array([], dtype=np.float64)
    sorter = np.argsort(a)
    inv = np.empty(n, dtype=np.intp)
    inv[sorter] = np.arange(n)

    a_sorted = a[sorter]
    obs = np.r_[True, a_sorted[1:] != a_sorted[:-1]]
    dense = obs.cumsum()[inv]

    count = np.r_[np.nonzero(obs)[0], n]
    return 0.5 * (count[dense] + count[dense - 1] + 1)

In [12]:
def compute_roc_auc(scores: np.ndarray, labels: np.ndarray) -> float:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)

    pos_scores = scores[labels == 1]
    neg_scores = scores[labels == 0]
    n_pos = len(pos_scores)
    n_neg = len(neg_scores)

    if n_pos == 0 or n_neg == 0:
        return 0.5

    all_scores = np.concatenate([pos_scores, neg_scores])
    all_labels = np.concatenate([np.ones(n_pos, dtype=np.int64), np.zeros(n_neg, dtype=np.int64)])
    ranks = _rank_data_numpy(all_scores)
    sum_pos_ranks = np.sum(ranks[all_labels == 1])
    u = sum_pos_ranks - (n_pos * (n_pos + 1.0)) / 2.0
    return float(u / (n_pos * n_neg))

In [13]:
def compute_per_graph_reconstruction_error(model, loader, config):
    model.eval()
    results = []
    feature_weights = torch.tensor(
        [1.0, 1.0, 1.0, 1.2, 2.0, 1.0, 1.5, 2.5, 2.5],
        device=DEVICE,
        dtype=torch.float32,
    )
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            if batch.num_graphs != 1:
                raise ValueError("Evaluation loader must use batch_size=1 to retain per-capture metadata.")

            outputs = model(
                x_stats=batch.x,
                id_idx=batch.id_idx,
                edge_index=batch.edge_index,
                edge_weight=batch.edge_attr,
            )
            node_se = (((outputs["x_recon"] - batch.x) ** 2) * feature_weights).mean(dim=-1)
            node_error = node_se.mean().item()

            if config.reconstruct_edges and outputs["edge_logits"] is not None and batch.edge_index.shape[1] > 0:
                pos_labels = torch.ones_like(outputs["edge_logits"])
                edge_loss = F.binary_cross_entropy_with_logits(outputs["edge_logits"], pos_labels).item()
                graph_error = node_error + 0.5 * edge_loss
            else:
                graph_error = node_error

            raw_label = batch.y[0].item()
            raw_capture = batch.capture_name
            if isinstance(raw_capture, (list, tuple)):
                capture_name = raw_capture[0]
            else:
                capture_name = raw_capture

            results.append((graph_error, raw_label, capture_name))

    return results

In [14]:
def main():
    set_seed(SEED)

    print("Loading graphs and vocab...")
    graphs = torch.load(GRAPH_DATA_PATH, weights_only=False)
    with open(VOCAB_PATH, "rb") as f:
        vocab = pickle.load(f)
    print(f"Loaded {len(graphs)} graphs, vocab size {len(vocab)}")

    config = ModelConfig()
    config.num_ids = len(vocab)

    align_graph_feature_dim(graphs, config.node_stat_feature_dim)
    train_graphs, val_graphs, test_graphs, train_caps, val_caps, test_caps = split_data(graphs)
    train_mean, train_std = normalize_graph_features(train_graphs, val_graphs, test_graphs)

    train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_graphs, batch_size=1, shuffle=False)

    print(f"Train-set feature stats: mean={train_mean[:4].tolist()} | std={train_std[:4].tolist()}")

    model = GraphTransformerAutoencoder(config).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print(f"Training on {DEVICE}...")
    best_val_loss = float("inf")

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer, config, train=True)
        val_loss = run_epoch(model, val_loader, optimizer, config, train=False)

        print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "model_state_dict": model.state_dict(),
                "config": config.__dict__,
                "epoch": epoch,
                "val_loss": val_loss,
                "train_mean": train_mean.cpu(),
                "train_std": train_std.cpu(),
                "train_captures": train_caps,
                "val_captures": val_caps,
                "test_captures": test_caps,
                "seed": SEED,
                "vocab": vocab,
            }, CHECKPOINT_PATH)

    print("\nEvaluating on test set (held-out benign + all attack windows)...")
    checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])

    graph_results = compute_per_graph_reconstruction_error(model, test_loader, config)
    errors = np.array([result[0] for result in graph_results], dtype=np.float32)
    labels = np.array([result[1] for result in graph_results], dtype=np.int64)

    per_capture_errors = {}
    for err, label, capture_name in graph_results:
        per_capture_errors.setdefault(capture_name, []).append((err, label))

    print("\nPer-capture reconstruction breakdown:")
    for capture_name in sorted(per_capture_errors):
        values = np.array([entry[0] for entry in per_capture_errors[capture_name]], dtype=np.float32)
        n_benign = sum(1 for entry in per_capture_errors[capture_name] if entry[1] == 0)
        n_attack = sum(1 for entry in per_capture_errors[capture_name] if entry[1] == 1)
        print(f"  {capture_name:45s}: mean={values.mean():.4f}, std={values.std():.4f}, "
              f"n_benign={n_benign}, n_attack={n_attack}")

    benign_errors = errors[labels == 0]
    attack_errors = errors[labels == 1]
    roc_auc = compute_roc_auc(errors, labels)

    print(f"\nAggregate Evaluation Summary:")
    print(f"  Benign reconstruction error : mean={benign_errors.mean():.4f}, std={benign_errors.std():.4f}, n={len(benign_errors)}")
    print(f"  Attack reconstruction error : mean={attack_errors.mean():.4f}, std={attack_errors.std():.4f}, n={len(attack_errors)}")
    print(f"  Reconstruction ROC-AUC      : {roc_auc:.4f}")

    if attack_errors.mean() > benign_errors.mean():
        print("\n-> Attack windows show HIGHER reconstruction error on average.")
    else:
        print("\n-> Attack windows do NOT show higher reconstruction error on average.")

In [16]:
if __name__ == "__main__":
    main()

Loading graphs and vocab...
Loaded 10996 graphs, vocab size 106
Split captures: 10 train ambient, 1 val ambient, 1 test ambient, 17 attack captures.
Split graphs: 7990 train (ambient), 2101 val (ambient), 905 test total.
Train-set feature stats: mean=[42.4364013671875, 0.3700965940952301, 0.006868728902190924, 1.7500449419021606] | std=[56.570865631103516, 0.45783138275146484, 0.039814870804548264, 1.653412103652954]
Training on cpu...
Epoch   1 | train_loss=106.2013 | val_loss=5.7037
Epoch   2 | train_loss=14.1096 | val_loss=4.2857
Epoch   3 | train_loss=9.6710 | val_loss=3.5714
Epoch   4 | train_loss=6.8356 | val_loss=3.2765
Epoch   5 | train_loss=5.3242 | val_loss=2.4386
Epoch   6 | train_loss=3.9883 | val_loss=2.4768
Epoch   7 | train_loss=3.1320 | val_loss=2.0142
Epoch   8 | train_loss=2.4376 | val_loss=1.7426
Epoch   9 | train_loss=1.8883 | val_loss=1.5063
Epoch  10 | train_loss=1.5445 | val_loss=1.2853
Epoch  11 | train_loss=1.2883 | val_loss=1.1237
Epoch  12 | train_loss=1.0846